In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from scipy import stats
from sklearn.preprocessing import LabelEncoder
import os

# CONFIGURAZIONE
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
xgb = XGBClassifier(n_estimators=100, eval_metric='mlogloss', random_state=42, n_jobs=-1)

# NOMI FILE
FILE_MEDIA = 'android_session_1_1.csv'
FILE_MEDIANA = 'android_session_1_1_median.csv'
FILE_FUSION = 'android_session_1_1_fusion_median.csv'

results = {}

print("AVVIO CALCOLO P-VALUE...")

# MEDIA vs MEDIANA
if os.path.exists(FILE_MEDIA) and os.path.exists(FILE_MEDIANA):
    print("\n[1/3] Calcolo Media vs Mediana...")
    df_mean = pd.read_csv(FILE_MEDIA).fillna(-110)
    df_median = pd.read_csv(FILE_MEDIANA).fillna(-110)

    cols_mean = [c for c in df_mean.columns if '-' in c]
    cols_median = [c for c in df_median.columns if '-' in c]

    scores_mean = cross_val_score(rf, df_mean[cols_mean], df_mean['artwork'], cv=cv, scoring='accuracy')
    scores_median = cross_val_score(rf, df_median[cols_median], df_median['artwork'], cv=cv, scoring='accuracy')

    t_stat, p_val = stats.ttest_rel(scores_mean, scores_median)
    results['Media_vs_Median'] = p_val
    results['Scores_Median'] = scores_median # Salvo per il prossimo step
    print(f"    P-Value: {p_val:.5e}")
else:
    print("    File mancanti per test 1.")

# RANDOM FOREST vs XGBOOST
if os.path.exists(FILE_MEDIANA):
    print("\n[2/3] Calcolo RF vs XGBoost...")

    df = pd.read_csv(FILE_MEDIANA).fillna(-110)
    cols = [c for c in df.columns if '-' in c]
    y = df['artwork']

    # XGBoost richiede target numerici
    le = LabelEncoder()
    y_enc = le.fit_transform(y)

    # Si utilizzano i punteggi RF calcolati sopra se esistono, altrimenti si ricalcolano
    if 'Scores_Median' in results:
        scores_rf = results['Scores_Median']
    else:
        scores_rf = cross_val_score(rf, df[cols], y, cv=cv, scoring='accuracy')

    scores_xgb = cross_val_score(xgb, df[cols], y_enc, cv=cv, scoring='accuracy')

    t_stat, p_val = stats.ttest_rel(scores_rf, scores_xgb)
    results['RF_vs_XGB'] = p_val
    print(f"    P-Value: {p_val:.5e}")
else:
    print("    File mancante per test 2.")

# RADIO vs FUSION
if os.path.exists(FILE_FUSION):
    print("\n[3/3] Calcolo Radio vs Fusion...")
    df_fus = pd.read_csv(FILE_FUSION).fillna(-110)

    cols_rssi = [c for c in df_fus.columns if '-' in c]
    cols_imu = [c for c in df_fus.columns if c not in cols_rssi and c not in ['room', 'artwork', 'timestamp', 'step_index', 'label']]

    X_radio = df_fus[cols_rssi]
    X_fusion = df_fus[cols_rssi + cols_imu]
    y = df_fus['artwork']

    scores_radio = cross_val_score(rf, X_radio, y, cv=cv, scoring='accuracy')
    scores_fusion = cross_val_score(rf, X_fusion, y, cv=cv, scoring='accuracy')

    t_stat, p_val = stats.ttest_rel(scores_radio, scores_fusion)
    results['Radio_vs_Fusion'] = p_val
    print(f"    P-Value: {p_val:.5e}")
else:
    print("    File Fusion mancante per test 3.")

print("\n" + "="*40)
print(" RISULTATI OTTENUTI ")
print("="*40)
if 'Media_vs_Median' in results:
    print(f"Sez 4.1.1 (Media vs Mediana): P-Value = {results['Media_vs_Median']:.2e}")
if 'RF_vs_XGB' in results:
    print(f"Sez 4.3.3 (RF vs XGBoost):    P-Value = {results['RF_vs_XGB']:.2e}")
if 'Radio_vs_Fusion' in results:
    print(f"Sez 4.13 (Radio vs Fusion):   P-Value = {results['Radio_vs_Fusion']:.2e}")